# SINCON 2026 핫식스 · 최종 전달 2단계 v2

1단계 `final_package/SINCON_final.ipynb`의 `RUN_MODE="assemble"` 출력 `submission_clean_refined.csv`를 input/에 놓습니다. 없으면 동봉 기준을 사용합니다. 이 노트북은 승인된 6개 층으로 주후보를 만들고, 선택 셀을 켜면 별도로 15행 후보를 만듭니다.

1단계 CSV는 기준 SHA가 같거나, 157,194행·고유 example_id·공식 순서·유한·비음수이고 동봉 기준과 최대 log1p 차이가 1e-9 이하일 때 허용합니다. 수치 동등성으로 허용하면 차이·다른 행 수·입력과 기준 SHA를 출력하고 verification.json에 기록합니다. 계산에는 제공된 CSV를 그대로 씁니다. 차이가 1e-9를 넘으면 중단하며 재학습 결과는 별도 검토가 필요합니다. 캐시·공식 CSV·동봉 기준의 해시는 엄격히 검사합니다. 6개 층을 켠 주후보와 선택 15행 후보는 각각 기준 대비 최대 log1p 차이 1e-10 이하여야 합니다.

공식 train.csv·test.csv·metaData.csv는 data/에 직접 준비합니다. 새 다운로드·Kaggle 업로드·Public 점수 조회 코드는 없습니다. 정답은 2026-06-30까지 공식 train뿐이며 7월 이후 지하철 실측을 찾거나 쓰지 않습니다. 설치는 노트북 밖에서 requirements.txt로 합니다. 전체 재학습의 상태는 1단계 STATUS를 확인하세요.

In [ ]:
# ROOT는 delivery_manifest.json이 있는 패키지 폴더입니다. 빈 값이면 제한 탐색합니다.
ROOT = ""
BASELINE_CSV = ""  # 패키지 기준 상대 경로 또는 절대 경로. 기본 input/ 우선, 없으면 reference/.
ENABLED = dict(timecapsule=True, plave_existing=True, station025=True, friday=True, plave22=True, lunar3122=True)
ENABLE_HEDGES = False  # 별도 15행 후보. True여도 주후보는 함께 저장합니다.

In [ ]:
import os
import sys
sys.dont_write_bytecode = True
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "1"
from pathlib import Path
import importlib.util
import json
from time import perf_counter

started = perf_counter()
def bounded_packages(root, max_depth, skip_names=()):
    found = []
    if root.is_dir():
        for directory, children, files in os.walk(root, followlinks=False):
            current = Path(directory)
            depth = len(current.relative_to(root).parts)
            children[:] = sorted(name for name in children if name not in skip_names
                                 and depth < max_depth and not (current / name).is_symlink())
            if "delivery_manifest.json" in files:
                found.append(current.resolve())
    return found

if ROOT:
    package = Path(ROOT).expanduser().resolve()
else:
    candidates = [base.resolve() for base in (Path.cwd(), Path.cwd() / "final_delivery_v2")
                  if (base / "delivery_manifest.json").is_file()]
    candidates += bounded_packages(Path("/kaggle/input"), 4)
    candidates += bounded_packages(Path("/content"), 3, ("drive",))
    candidates = list(dict.fromkeys(candidates))
    if len(candidates) != 1:
        raise ValueError("ROOT에 패키지 폴더를 지정하세요. /content 깊이3·drive 제외, /kaggle/input 깊이4 탐색. 발견 수: " + str(len(candidates)))
    package = candidates[0]
if not (package / "delivery_manifest.json").is_file():
    raise FileNotFoundError("ROOT는 delivery_manifest.json이 있는 패키지 폴더여야 합니다")
spec = importlib.util.spec_from_file_location("sincon_delivery_runtime", package / "delivery_runtime.py")
rt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rt)
context = rt.prepare(package, BASELINE_CSV)
print("패키지:", package)
print("공식 CSV:", context["data"])
print("1단계 입력:", context["baseline"].name)
print(json.dumps(context["environment"], ensure_ascii=False, indent=2))

## 승인된 순서와 스위치

| 순서 | 스위치 | 대상 | 작동 |
|---|---|---|---|
| 1 | timecapsule | 7/11 송도달빛축제공원 21~23시 3행 | 같은 시리즈 과거 초과분 75% 하한 |
| 2 | plave_existing | 9/12~13 문학 21~22시 4행 | 과거 공연 합계로 log 차이 35%, cap .50 |
| 3 | station025 | 전체 역 | 최근 60일 역별 평균 오차를 중심화해 .25배 차감 |
| 4 | friday | 7/17·10/9 2,016행 | 역간 상대 차이, 시간별 중심화, log ±.05 |
| 5 | plave22 | 9/12~13 문학 22시 2행 | 평소+과거 22시 비율×지정 예매 |
| 6 | lunar3122 | 9/24~25 부평삼거리 36행 | 평소 일요일 대비 전날·당일 전이 |

ENABLED의 한 줄로 층을 끌 수 있습니다. 꺼진 조합은 EXPERIMENT_layers_disabled.csv이며 주후보로 표시하지 않습니다. 앞선 행사층 스위치에 따라 역별 오차도 다시 적합합니다. 수치 계산·계수는 v1 그대로이고 final_layers.py에서는 1단계 입력 검사와 입력 해시 기록만 바꿨습니다.

보관 원시 예측에서 기준일마다 그 시점까지의 정답만으로 층 채택을 다시 정한 순방향 OOF를 만들었다. 최근 60일·0.25배 역별 층은 유효 8개 기준일 중 7개 개선했고 B는 악화했다. 금요일 층은 발동 5개 기준일에서, 부평삼거리 층은 발동 6개 기준일에서 개선했다. 기준일별 평가기간이 겹치며 금요일은 두 날짜, 명절은 두 회차다. 기존 OOF는 최종 채택 구성을 소급한 조건부 재계산으로 구분한다. 최종 역별 보정은 두 적합법의 최대 log 차이가 0.000067407이어서 기존값을 유지했다.

In [ ]:
main, deltas, layer_report = context["layers_module"].build_submission(
    context["data"], package / "cache", context["baseline"],
    enabled={name for name, enabled in ENABLED.items() if enabled},
)
print(rt.layer_table(deltas, layer_report).to_string(index=False))

## 1. 누락 타임캡슐 3행

7/11 누락 공연에 2025-06-14 같은 시리즈의 초과분 75% 하한을 적용합니다. 21~23시 실제 272/489/88, 평소 80/78/55, 목표 평소 102/77/58로 역별 보정 전 하한 246/385.25/82.75입니다.

보관 ROUND2의 summer 36행 RMSLE 1.697556589→1.414320016, 7/31 origin의 같은 36행 1.726618182→1.399029261입니다. 같은 12일의 반복 검증이며 같은 시리즈 기증은 1건입니다. 새 독립 홀드아웃은 아닙니다.

In [ ]:
rt.show_layer("timecapsule", deltas, layer_report)

## 2. 기존 PLAVE 4행

완결된 공연의 21·22시 합계를 계열별 중앙값과 계열 간 중앙값으로 모읍니다. 현재 합계와 log 차이 .35배를 두 행에 적용하고 0~.50으로 제한합니다. 보관 ROUND2의 4계열 14행 RMSLE 2.079960524→1.657658006, 13행 개선입니다. 작은 부분집합이며 나중 22시 대체와 단순 합산할 수 없습니다. 과거 예측 비교가 가능한 공연 날짜는 고유 7일이며 2025-03-20·21은 OOF가 없습니다.

In [ ]:
rt.show_layer("plave_existing", deltas, layer_report)

## 3. 역별 최근 오차 60일·0.25배

최종 적합은 2026-05-02~06-30, 최신 origin 2026-04-30입니다. 230,040개 예측 잔차에 고유 실측 키 76,680개가 반복 포함됩니다. 역별 평균 오차를 중심화해 .25배 차감하고 360행 이상인 역만 적용합니다. 공식 train으로 OOF 정답을 다시 연결합니다.

| origin | 구분 | 중첩 기준 RMSLE | 중첩 적용 RMSLE | Δ |
|---|---|---|---|---|
| 2025-06-30 | summer | 0.253533140 | 0.253533140 | 0.000000000 |
| 2025-07-31 | 2025-07-31 | 0.179176853 | 0.174839821 | -0.004337032 |
| 2025-08-31 | 2025-08-31 | 0.176996445 | 0.175134740 | -0.001861705 |
| 2025-09-30 | 2025-09-30 | 0.171653974 | 0.169531901 | -0.002122073 |
| 2025-10-31 | C | 0.161397539 | 0.159811426 | -0.001586113 |
| 2025-12-31 | B | 0.164613239 | 0.166184612 | 0.001571373 |
| 2026-02-28 | A | 0.150075145 | 0.149683397 | -0.000391749 |
| 2026-03-31 | 2026-03-31 | 0.148569348 | 0.148319964 | -0.000249384 |
| 2026-04-30 | 2026-04-30 | 0.143385799 | 0.142617509 | -0.000768290 |

중첩 OOF 유효 8개 기준일 중 7개 개선·B 악화입니다. summer는 이전 OOF가 없어 보정하지 않습니다. 기존값은 최종 채택 구성을 소급한 조건부 재계산으로 VALIDATION.md에 분리했습니다. 두 적합법의 최대 log 보정 차이는 .000067407이며 승인된 기존 최종값을 유지합니다.

In [ ]:
rt.show_layer("station025", deltas, layer_report)

## 4. 금요일 공휴일 2,016행

주변 ±42일 평소 금요일 대비 과거 금요일 3일 연휴의 차이를 구합니다. 시간별 역간 평균을 0으로 맞춰 .20배 적용하고 log ±.05로 제한합니다. log .05는 1+승차 비율 약 +5.13%입니다. 행사 제외 15역과 보호 키를 유지합니다.

| origin | 노출 날짜 | 행 | 중첩 기준 RMSLE | 중첩 적용 RMSLE | Δ |
|---|---|---|---|---|---|
| 2025-06-30 | 2025-08-15 | 1008 | 0.218975464 | 0.216518433 | -0.002457031 |
| 2025-07-31 | 2025-08-15 | 1008 | 0.183296996 | 0.179744253 | -0.003552743 |
| 2026-02-28 | 2026-05-01 | 1008 | 0.156459318 | 0.155706314 | -0.000753004 |
| 2026-03-31 | 2026-05-01 | 1008 | 0.149287782 | 0.148296236 | -0.000991546 |
| 2026-04-30 | 2026-05-01 | 1008 | 0.149551696 | 0.148457976 | -0.001093720 |

표는 중첩 OOF의 실제 변경 1,008행입니다. 5개 기준일 모두 개선했지만 고유 날짜는 2개입니다. 최종 목표 7/17·10/9는 각 1,008행입니다. 전체 기간 표와 기존 조건부 재계산은 VALIDATION.md에 있습니다.

In [ ]:
rt.show_layer("friday", deltas, layer_report)

## 5. PLAVE 22시 2행

과거 계열별 `(22시 실제−평소)/관객 대용치`의 Q25·Q50 기하평균 .173226702067872를 씁니다. 토 평소 21.5+비율×23,120, 일 평소 15+비율×19,494입니다. 선택 후보에서도 두 행은 유지합니다.

보관 ROUND5의 22시 5행에서 OOF/기존식 재구성/시간 순서 제안 RMSLE는 2.572896/2.083644/.589066입니다. 관객을 안다는 조건의 작은 부분집합이며 현재 전체 후보의 정확한 OOF가 아닙니다. 6행 시간 배분은 시간순 검증에서 실패해 제외했습니다. 버스 비중은 계수 계산에 쓰지 않았고 예매 원출처·이용 조건은 미확인입니다.

In [ ]:
rt.show_layer("plave22", deltas, layer_report)

## 6. 부평삼거리 전날·당일 36행

주변 ±35일 평소 일요일 목표 예측 log 중앙값에 과거 명절 같은 상대일의 실제 log 차이 평균을 더합니다. 기증일과 그 주변 일요일 실측은 cutoff 이하로 자릅니다. 최종 9/24·25의 18시간씩만 대체합니다.

| origin | 구분 | 변경 행 | 중첩 기준 RMSLE | 중첩 적용 RMSLE | Δ |
|---|---|---|---|---|---|
| 2025-06-30 | summer | 36 | 0.253519608 | 0.253228249 | -0.000291359 |
| 2025-07-31 | 2025-07-31 | 36 | 0.174815973 | 0.174278267 | -0.000537706 |
| 2025-08-31 | 2025-08-31 | 36 | 0.175134740 | 0.174580606 | -0.000554134 |
| 2025-09-30 | 2025-09-30 | 36 | 0.169531901 | 0.169006703 | -0.000525198 |
| 2025-10-31 | C | 36 | 0.159811426 | 0.159681316 | -0.000130111 |
| 2025-12-31 | B | 36 | 0.166184612 | 0.165983736 | -0.000200876 |

중첩 OOF 발동 6개 모두 개선했습니다. 고유 수령 명절은 2회이고 216 OOF행은 고유 키 72개의 반복입니다. station025·friday 뒤이며 PLAVE22를 OOF 전체에 붙인 점수는 아닙니다. 옛 미래 일요일 혼입을 제거해도 최종값은 같았습니다. 당일 금요일의 직접 과거 검증은 없고 시간 묶음은 C/B에서 악화해 제외했습니다.

In [ ]:
rt.show_layer("lunar3122", deltas, layer_report)

## 선택 셀 · 별도 15행 후보

ENABLE_HEDGES=True이면 submission_night_v5_events_hedged15.csv를 추가로 만듭니다. 주후보와 지정된 15행만 달라야 합니다. 비교 기준 SHA-256은 9b01382c398fa49c141c78da278e678e7930a75373a8dd1dafa1c2d076c74b6c입니다.

PLAVE는 21시 10%·23시 5%만 계산하고 22시는 주후보를 유지합니다. Sustainable은 가정 관객 12,000명, 20/21/22/23시 10%/20%/45%/25%, W=.5입니다. 랜더스 FULL은 기존 2,199/4,102/860, W=.5입니다. PLAVE W=.7을 포함해 계수는 새로 정하지 않았습니다. 혼합식은 `(1+현재)^(1−W) × (1+FULL)^W − 1`입니다. 평소 표는 반올림한 기존 정수입니다.

11번 판정은 지지 2행·근거 부족 13행입니다. 지지는 랜더스 21·22시의 상향 방향과 과거 규모만을 뜻하며 W의 독립 검증이 아닙니다. 시험기간 정답에 대한 성능은 미평가입니다. 박효신은 시작 시각과 실제 시간 비중만 참고했고 공연 길이는 미확인입니다. 2026-03-28~06-30 야구 36경기는 관중 기록이 있는 전 경기이며 매진은 그중 11경기입니다.

| ID | 행사 | 날짜 | 시 | 주후보 | 15행 후보 값 | FULL | W | 11번 판정 |
|---|---|---|---|---|---|---|---|---|
| J093669 | PLAVE | 2026-09-12 | 21 | 1111.760411 | 849.214439 | 756.600000 | 0.7 | 근거 부족 |
| J093671 | PLAVE | 2026-09-12 | 23 | 233.924861 | 311.283255 | 351.800000 | 0.7 | 근거 부족 |
| J094947 | PLAVE | 2026-09-13 | 21 | 762.978309 | 651.477399 | 608.820000 | 0.7 | 근거 부족 |
| J094949 | PLAVE | 2026-09-13 | 23 | 147.014071 | 240.234711 | 296.410000 | 0.7 | 근거 부족 |
| J075776 | Sustainable Wave Festival | 2026-08-29 | 20 | 223.175830 | 337.458046 | 510.000000 | 0.5 | 근거 부족 |
| J075777 | Sustainable Wave Festival | 2026-08-29 | 21 | 443.805337 | 589.531442 | 783.000000 | 0.5 | 근거 부족 |
| J075778 | Sustainable Wave Festival | 2026-08-29 | 22 | 408.637640 | 819.137187 | 1641.000000 | 0.5 | 근거 부족 |
| J075779 | Sustainable Wave Festival | 2026-08-29 | 23 | 68.362851 | 249.684548 | 905.000000 | 0.5 | 근거 부족 |
| J077054 | Sustainable Wave Festival | 2026-08-30 | 20 | 149.344579 | 243.616317 | 397.000000 | 0.5 | 근거 부족 |
| J077055 | Sustainable Wave Festival | 2026-08-30 | 21 | 254.903529 | 435.632716 | 744.000000 | 0.5 | 근거 부족 |
| J077056 | Sustainable Wave Festival | 2026-08-30 | 22 | 236.325543 | 622.108810 | 1635.000000 | 0.5 | 근거 부족 |
| J077057 | Sustainable Wave Festival | 2026-08-30 | 23 | 40.818054 | 193.538785 | 904.000000 | 0.5 | 근거 부족 |
| J138398 | 랜더스페셜 매치 | 2026-10-17 | 20 | 788.997981 | 1317.330595 | 2199.000000 | 0.5 | 근거 부족 |
| J138399 | 랜더스페셜 매치 | 2026-10-17 | 21 | 1169.265345 | 2190.255054 | 4102.000000 | 0.5 | 지지 |
| J138400 | 랜더스페셜 매치 | 2026-10-17 | 22 | 285.907167 | 496.018180 | 860.000000 | 0.5 | 지지 |

값은 표시용 소수 6자리입니다. 계산은 반올림하지 않으며 전체 정밀도와 근거·한계는 evidence/hedges_15_current_review.csv에 있습니다.

In [ ]:
if ENABLE_HEDGES:
    optional_candidate, assumptions = rt.apply_hedges(main, context["grid"], package)
    print(assumptions.to_string(index=False))
else:
    print("선택 셀 꺼짐. 별도 15행 후보는 W·시간 비중의 독립 검증이 없는 추정입니다.")

## 저장 후 검증

형식, ID 순서·고유성, 유한·비음수, 저장 후 값 보존을 검사합니다. 6개 층 주후보와 선택 후보는 각 동봉 기준 대비 최대 log1p 차이 1e-10 이하를 강제합니다. 15행의 정확한 변경 ID도 검사합니다. 입력을 수치 동등성으로 받아들인 내역은 verification.json의 baseline_verification에 남습니다.

매 실행은 새 시각 폴더입니다. Kaggle /kaggle/working, Colab /content, 로컬 outputs/를 사용합니다. SHA는 바이트 식별자이며 미래 정확도가 아닙니다.

In [ ]:
result = rt.finish(context, main, deltas, layer_report, enable_hedges=ENABLE_HEDGES)
print("전체 코드 셀 계산 경과 초:", round(perf_counter() - started, 3))

## 이력·넣지 않은 것·확인 범위

17행 버전은 이력입니다. PLAVE 22시 두 행을 다시 올리며 11번 판정은 그 두 행이 과함, 나머지가 지지 2·근거 부족 13입니다. 7번 옛 표는 PLAVE 3%/85%/12%, 랜더스 FULL 3,255/3,513/366을 사용한 당시 검토여서 현재 판정으로 인용하지 않습니다. 17행 기준 CSV·현재 17행 검토는 evidence/history/, 7번 원 판정은 evidence/hedges_17_judgments_round7_original.csv에 보존했습니다.

12번의 모델층 증폭 규칙 세 가지는 모두 통과하지 못해 후보에 넣지 않았습니다.

출처 조건은 SOURCES.md, 감사9번 문서 교정은 DOCUMENT_CORRECTIONS.md, 실제 실행은 STATUS.md와 verification/을 확인하세요. Claude는 v1 ZIP의 표준 Jupyter kernel에서 주후보·17행 후보를 바뀐 행 0으로 확인했고, 1단계 Python 3.11 실환경의 표준 kernel에서는 수치 일치·바이트 불일치를 확인했습니다. 이는 Claude 검증이며 이번 Codex 셀 실행기 검사와 구분합니다. Kaggle·Colab·Linux 실기는 미실행입니다. 전체 모델 재학습·권리 확인 완료를 뜻하지 않습니다.